<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/PAPSS2_Top1_Oldipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PAPSS2 OLD DERIVATIVES — TOP 10 EVALUATION
# CELL 1: UPLOAD AND INPUT AUDIT
# ============================================================

!pip -q install rdkit openpyxl

from google.colab import files
import pandas as pd
from rdkit import Chem
import os

uploaded = files.upload()

# Identify input files
excel_file = "PAPSS2_Top10_Old_Derivative_docked.xlsx"
sdf_file = "PAPSS2_Top10_Old_Derivative.sdf"

assert excel_file in uploaded, f"Missing Excel file: {excel_file}"
assert sdf_file in uploaded, f"Missing SDF file: {sdf_file}"

# Load Excel
df = pd.read_excel(excel_file)

# Load SDF
supplier = Chem.SDMolSupplier(sdf_file, removeHs=False)
molecules = [mol for mol in supplier if mol is not None]

print("========== INPUT AUDIT ==========")
print("Excel dimensions:", df.shape)
print("SDF valid structures:", len(molecules))
print("Excel columns:", df.columns.tolist())

# Check identifiers
excel_cids = set(df["PUBCHEM CID"].dropna().astype(int))
sdf_cids = {
    int(mol.GetProp("PUBCHEM_CID"))
    for mol in molecules
    if mol.HasProp("PUBCHEM_CID")
}

print("\nExcel candidate CIDs:", len(excel_cids))
print("SDF candidate CIDs:", len(sdf_cids))
print("Matching CIDs:", len(excel_cids & sdf_cids))
print("Excel-only CIDs:", excel_cids - sdf_cids)
print("SDF-only CIDs:", sdf_cids - excel_cids)

# Candidate overview
display(
    df[[
        "GLOBAL RANK",
        "PUBCHEM CID",
        "IUPAC NAME",
        "EDOCK",
        "SOURCE LIBRARY",
        "STRUCTURE NOTE"
    ]]
)

assert len(df) == 10, "Expected exactly 10 Excel candidates."
assert len(molecules) == 10, "Expected exactly 10 valid SDF structures."
assert excel_cids == sdf_cids, "Excel and SDF CID mismatch."

print("\nAUDIT PASSED: All 10 candidates have matching SDF structures.")

Saving PAPSS2_Top10_Old_Derivative.sdf to PAPSS2_Top10_Old_Derivative.sdf
Saving PAPSS2_Top10_Old_Derivative_docked.xlsx to PAPSS2_Top10_Old_Derivative_docked.xlsx
========== INPUT AUDIT ==========
Excel dimensions: (10, 18)
SDF valid structures: 10
Excel columns: ['mol', 'L', 'IX', 'NAME', 'Edock', 'Egb', 'Ege', 'Egs', 'Egv', 'CONF', 'RecConf', 'Molecule Name', 'PUBCHEM CID', 'IUPAC NAME', 'GLOBAL RANK', 'EDOCK', 'STRUCTURE NOTE', 'SOURCE LIBRARY']

Excel candidate CIDs: 10
SDF candidate CIDs: 10
Matching CIDs: 10
Excel-only CIDs: set()
SDF-only CIDs: set()


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,EDOCK,SOURCE LIBRARY,STRUCTURE NOTE
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",-84.31694,Adamantane,Generated 2D structure; not an ICM-Pro docked ...
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,-83.90470,Piperidines,Generated 2D structure; not an ICM-Pro docked ...
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",-80.23081,Piperidines,Generated 2D structure; not an ICM-Pro docked ...
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,-74.13148,Piperidines,Generated 2D structure; not an ICM-Pro docked ...
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",-65.49297,Phytochemicals,Generated 2D structure; not an ICM-Pro docked ...
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",-61.87345,Phytochemicals,Generated 2D structure; not an ICM-Pro docked ...
6,7,10983,2-(3-methylbut-2-enyl)guanidine,-61.05599,Phytochemicals,Generated 2D structure; not an ICM-Pro docked ...
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",-60.69700,Phytochemicals,Generated 2D structure; not an ICM-Pro docked ...
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,-57.70152,Piperidines,Generated 2D structure; not an ICM-Pro docked ...
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,-57.40389,Piperidines,Generated 2D structure; not an ICM-Pro docked ...



AUDIT PASSED: All 10 candidates have matching SDF structures.


In [2]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 2: MOLECULAR DESCRIPTOR CALCULATION
# ============================================================

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED, Crippen, rdMolDescriptors
import pandas as pd

results = []

for mol in molecules:

    cid = int(mol.GetProp("PUBCHEM_CID"))

    row = df.loc[df["PUBCHEM CID"].astype(int) == cid].iloc[0]

    results.append({
        "GLOBAL RANK": int(row["GLOBAL RANK"]),
        "PUBCHEM CID": cid,
        "COMPOUND NAME": row["Molecule Name"],
        "IUPAC NAME": row["IUPAC NAME"],
        "SOURCE LIBRARY": row["SOURCE LIBRARY"],
        "EDOCK": float(row["EDOCK"]),
        "Molecular Weight": Descriptors.MolWt(mol),
        "Exact Molecular Weight": Descriptors.ExactMolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "Rotatable Bonds": Lipinski.NumRotatableBonds(mol),
        "Heavy Atoms": Lipinski.HeavyAtomCount(mol),
        "Aromatic Rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "QED": QED.qed(mol),
        "Lipinski Violations": sum([
            Descriptors.MolWt(mol) > 500,
            Crippen.MolLogP(mol) > 5,
            Lipinski.NumHDonors(mol) > 5,
            Lipinski.NumHAcceptors(mol) > 10
        ])
    })

descriptor_df = pd.DataFrame(results).sort_values("GLOBAL RANK")

print("Descriptor calculation completed.")
print("Candidates evaluated:", len(descriptor_df))

display(descriptor_df.round(3))

# Save intermediate output
descriptor_df.to_excel(
    "PAPSS2_Old_Top10_RDKit_Descriptors.xlsx",
    index=False
)

print("Saved: PAPSS2_Old_Top10_RDKit_Descriptors.xlsx")

Descriptor calculation completed.
Candidates evaluated: 10


,GLOBAL RANK,PUBCHEM CID,COMPOUND NAME,IUPAC NAME,SOURCE LIBRARY,EDOCK,Molecular Weight,Exact Molecular Weight,LogP,TPSA,HBD,HBA,Rotatable Bonds,Heavy Atoms,Aromatic Rings,QED,Lipinski Violations
0,1,64148,CID_64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Adamantane,-84.317,240.266,239.999,-2.445,81.24,0,4,0,14,0,0.471,0
1,2,44263409,CID_44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,Piperidines,-83.905,518.547,518.218,3.837,125.40,3,7,14,36,2,0.221,1
2,3,11181229,CID_11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",Piperidines,-80.231,636.726,636.296,6.229,125.40,3,7,18,45,3,0.126,2
3,4,455458,CID_455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,Piperidines,-74.131,376.633,376.357,3.277,12.96,0,4,6,27,0,0.705,0
4,5,5281148,CID_5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",Phytochemicals,-65.493,260.377,260.178,2.818,40.46,2,2,10,19,0,0.400,0
5,6,5281149,CID_5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",Phytochemicals,-61.873,244.378,244.183,3.847,20.23,1,1,9,18,0,0.391,0
6,7,10983,CID_10983,2-(3-methylbut-2-enyl)guanidine,Phytochemicals,-61.056,127.191,127.111,0.226,64.40,2,1,2,9,0,0.317,0
7,8,393472,CID_393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",Phytochemicals,-60.697,470.647,470.324,3.550,116.45,4,7,21,33,0,0.190,0
8,9,9919108,CID_9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,Piperidines,-57.702,861.562,860.507,6.389,136.82,1,9,28,60,2,0.059,2
9,10,11628768,CID_11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,Piperidines,-57.404,627.776,627.306,-0.636,274.25,8,7,14,44,2,0.098,2


Saved: PAPSS2_Old_Top10_RDKit_Descriptors.xlsx


In [3]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 3: ADMET-AI PREDICTION
# ============================================================

!pip -q install admet-ai

from admet_ai import ADMETModel
from rdkit import Chem
import pandas as pd

# Prepare structures and identifiers
admet_input = []

for mol in molecules:
    cid = int(mol.GetProp("PUBCHEM_CID"))
    row = df.loc[df["PUBCHEM CID"].astype(int) == cid].iloc[0]

    admet_input.append({
        "PUBCHEM CID": cid,
        "GLOBAL RANK": int(row["GLOBAL RANK"]),
        "COMPOUND NAME": row["IUPAC NAME"],
        "SMILES": Chem.MolToSmiles(mol)
    })

admet_input_df = pd.DataFrame(admet_input)

# Initialize ADMET-AI
model = ADMETModel()

# Predict
smiles_list = admet_input_df["SMILES"].tolist()
admet_predictions = model.predict(smiles=smiles_list)

# Convert predictions to DataFrame
admet_df = pd.DataFrame(admet_predictions)

# Attach candidate identifiers
admet_df.insert(
    0, "COMPOUND NAME", admet_input_df["COMPOUND NAME"].values
)
admet_df.insert(
    0, "GLOBAL RANK", admet_input_df["GLOBAL RANK"].values
)
admet_df.insert(
    0, "PUBCHEM CID", admet_input_df["PUBCHEM CID"].values
)

# Save complete endpoint output
admet_df.to_excel(
    "PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.xlsx",
    index=False
)

print("ADMET-AI prediction completed.")
print("Candidates:", len(admet_df))
print("Predicted endpoints:", len(admet_df.columns) - 3)

display(admet_df)

print("Saved: PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.xlsx")

model ensembles:   0%|          | 0/2 [00:00<?, ?it/s]INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()

Output()

Output()

Output()

Output()

Output()

... (6 outputs left out)

model ensembles: 100%|██████████| 2/2 [00:01<00:00,  1.96it/s]

ADMET-AI prediction completed.
Candidates: 10
Predicted endpoints: 104


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,molecular_weight,logP,hydrogen_bond_acceptors,hydrogen_bond_donors,Lipinski,QED,stereo_centers,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
[H]C1([H])N2C([H])([H])N3C([H])([H])N(C([H])([H])N1S3(=O)=O)S2(=O)=O,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",240.266,-2.44520,4,0,4.0,0.470678,0,...,46.362039,59.753954,65.483304,15.254833,74.376098,52.724077,37.188049,25.131810,78.137083,5.659051
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])([H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,518.547,3.83720,5,3,3.0,0.221491,2,...,16.098418,59.437610,73.954306,17.539543,26.291740,59.332162,71.142355,81.933216,56.977153,45.096661
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])(C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",636.726,6.22870,5,3,2.0,0.125790,2,...,15.746924,58.664323,81.335677,54.059754,26.572935,75.219684,85.377856,92.618629,48.049209,30.263620
[H]C([H])(N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H])C([H])([H])C([H])([H])N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H],455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,376.633,3.27710,4,0,4.0,0.705153,0,...,26.151142,10.544815,8.506151,77.539543,42.636204,78.347979,19.683656,3.374341,82.811951,94.059754
[H]O[C@]([H])(C#CC#C[C@]([H])(O[H])C([H])=C([H])[H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H],5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",260.377,2.81770,2,2,4.0,0.400102,2,...,66.713533,44.499121,42.530756,73.673111,72.091388,86.959578,68.506151,66.854130,40.175747,86.502636
[H]O[C@@]([H])(C#CC#CC([H])([H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])=C([H])[H],5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",244.378,3.84690,1,1,4.0,0.390506,1,...,87.908612,70.896309,66.151142,84.463972,89.666081,85.694200,86.924429,76.485062,14.938489,91.634446
[H]C(=C(C([H])([H])[H])C([H])([H])[H])C([H])([H])N=C(N([H])[H])N([H])[H],10983,7,2-(3-methylbut-2-enyl)guanidine,127.191,0.22600,1,2,4.0,0.316543,0,...,36.239016,44.956063,26.537786,7.838313,51.845343,46.467487,25.834798,8.857645,85.483304,17.750439
[H]OC([H])(C([H])([H])C1=C([H])C([H])(C([H])([H])[H])OC1=O)C([H])([H])C([H])([H])C([H])([H])C([H])([H])C1([H])OC([H])(C([H])(O[H])C([H])([H])C([H])([H])C([H])(O[H])C([H])(O[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C1([H])[H],393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",470.647,3.55040,7,4,4.0,0.189702,7,...,56.133568,81.405975,75.641476,63.901582,36.942004,31.845343,65.377856,43.339192,40.492091,82.741652
[Cl-].[H]c1c([H])c([H])c(C(=O)N(C(=O)OC([H])([H])C([H])(OC([H])([H])[H])C([H])([H])OC(=O)N2C([H])([H])C([H])([H])C([H])(OC(=O)N([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C2([H]

Saved: PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.xlsx


In [4]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 4: ADMET ENDPOINT AUDIT
# ============================================================

# Separate prediction columns from candidate identifiers
metadata_cols = [
    "PUBCHEM CID",
    "GLOBAL RANK",
    "COMPOUND NAME"
]

endpoint_cols = [
    col for col in admet_df.columns
    if col not in metadata_cols
]

# Endpoint inventory
endpoint_inventory = pd.DataFrame({
    "Endpoint": endpoint_cols,
    "Endpoint_Type": [
        "Percentile" if "percentile" in col.lower()
        else "Prediction"
        for col in endpoint_cols
    ]
})

print("Total ADMET columns:", len(endpoint_cols))
print("\nEndpoint type distribution:")
print(endpoint_inventory["Endpoint_Type"].value_counts())

print("\nCOMPLETE ENDPOINT INVENTORY:")
display(endpoint_inventory)

# Save endpoint inventory
endpoint_inventory.to_excel(
    "PAPSS2_Old_Top10_ADMET_Endpoint_Inventory.xlsx",
    index=False
)

# Save complete predictions in CSV format too
admet_df.to_csv(
    "PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.csv",
    index=False
)

print("\nSaved:")
print("1. PAPSS2_Old_Top10_ADMET_Endpoint_Inventory.xlsx")
print("2. PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.csv")

Total ADMET columns: 104

Endpoint type distribution:
Endpoint_Type
Prediction    52
Percentile    52
Name: count, dtype: int64

COMPLETE ENDPOINT INVENTORY:


,Endpoint,Endpoint_Type
0,molecular_weight,Prediction
1,logP,Prediction
2,hydrogen_bond_acceptors,Prediction
3,hydrogen_bond_donors,Prediction
4,Lipinski,Prediction
...,...,...
99,LD50_Zhu_drugbank_approved_percentile,Percentile
100,Lipophilicity_AstraZeneca_drugbank_approved_pe...,Percentile
101,PPBR_AZ_drugbank_approved_percentile,Percentile
102,Solubility_AqSolDB_drugbank_approved_percentile,Percentile



Saved:
1. PAPSS2_Old_Top10_ADMET_Endpoint_Inventory.xlsx
2. PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.csv


In [5]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 5: ADMET ENDPOINT SUMMARY
# ============================================================

# Display all 52 primary prediction endpoints
prediction_endpoints = [
    c for c in endpoint_cols
    if not c.endswith("_percentile")
]

percentile_endpoints = [
    c for c in endpoint_cols
    if c.endswith("_percentile")
]

print("Primary prediction endpoints:", len(prediction_endpoints))
print("Reference percentile endpoints:", len(percentile_endpoints))

# Create candidate-wise prediction summary
prediction_summary = admet_df[
    metadata_cols + prediction_endpoints
].copy()

# Create candidate-wise percentile summary
percentile_summary = admet_df[
    metadata_cols + percentile_endpoints
].copy()

# Save separately
prediction_summary.to_excel(
    "PAPSS2_Old_Top10_ADMET_Primary_Predictions.xlsx",
    index=False
)

percentile_summary.to_excel(
    "PAPSS2_Old_Top10_ADMET_Reference_Percentiles.xlsx",
    index=False
)

# Print complete endpoint names for interpretation
print("\nPRIMARY PREDICTION ENDPOINTS")
for i, endpoint in enumerate(prediction_endpoints, 1):
    print(f"{i}. {endpoint}")

print("\nREFERENCE PERCENTILE ENDPOINTS")
for i, endpoint in enumerate(percentile_endpoints, 1):
    print(f"{i}. {endpoint}")

print("\nSaved both endpoint summary workbooks.")

Primary prediction endpoints: 52
Reference percentile endpoints: 52

PRIMARY PREDICTION ENDPOINTS
1. molecular_weight
2. logP
3. hydrogen_bond_acceptors
4. hydrogen_bond_donors
5. Lipinski
6. QED
7. stereo_centers
8. tpsa
9. PAINS_alert
10. BRENK_alert
11. NIH_alert
12. AMES
13. BBB_Martins
14. Bioavailability_Ma
15. CYP1A2_Veith
16. CYP2C19_Veith
17. CYP2C9_Substrate_CarbonMangels
18. CYP2C9_Veith
19. CYP2D6_Substrate_CarbonMangels
20. CYP2D6_Veith
21. CYP3A4_Substrate_CarbonMangels
22. CYP3A4_Veith
23. Carcinogens_Lagunin
24. ClinTox
25. DILI
26. HIA_Hou
27. NR-AR-LBD
28. NR-AR
29. NR-AhR
30. NR-Aromatase
31. NR-ER-LBD
32. NR-ER
33. NR-PPAR-gamma
34. PAMPA_NCATS
35. Pgp_Broccatelli
36. SR-ARE
37. SR-ATAD5
38. SR-HSE
39. SR-MMP
40. SR-p53
41. Skin_Reaction
42. hERG
43. Caco2_Wang
44. Clearance_Hepatocyte_AZ
45. Clearance_Microsome_AZ
46. Half_Life_Obach
47. HydrationFreeEnergy_FreeSolv
48. LD50_Zhu
49. Lipophilicity_AstraZeneca
50. PPBR_AZ
51. Solubility_AqSolDB
52. VDss_Lombardo

REF

In [6]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 6: ADMET CLASSIFICATION AND RISK TABLES
# ============================================================

import pandas as pd

# Binary classification endpoints
toxicity_endpoints = [
    "AMES", "Carcinogens_Lagunin", "ClinTox", "DILI",
    "Skin_Reaction", "hERG",
    "NR-AR-LBD", "NR-AR", "NR-AhR", "NR-Aromatase",
    "NR-ER-LBD", "NR-ER", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"
]

cyp_endpoints = [
    "CYP1A2_Veith", "CYP2C19_Veith",
    "CYP2C9_Veith", "CYP2D6_Veith",
    "CYP3A4_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP3A4_Substrate_CarbonMangels"
]

absorption_endpoints = [
    "BBB_Martins", "Bioavailability_Ma",
    "HIA_Hou", "PAMPA_NCATS",
    "Pgp_Broccatelli"
]

# Check endpoint availability
for endpoint in toxicity_endpoints + cyp_endpoints + absorption_endpoints:
    assert endpoint in admet_df.columns, f"Missing endpoint: {endpoint}"

# Candidate metadata
candidate_info = admet_df[
    ["PUBCHEM CID", "GLOBAL RANK", "COMPOUND NAME"]
].copy()

# Separate endpoint tables
toxicity_table = candidate_info.join(
    admet_df[toxicity_endpoints]
)

cyp_table = candidate_info.join(
    admet_df[cyp_endpoints]
)

absorption_table = candidate_info.join(
    admet_df[absorption_endpoints]
)

# Generate transparent preliminary probability flags
# >= 0.5 is a screening convention, NOT a validated clinical cutoff.

toxicity_flags = candidate_info.copy()

for endpoint in toxicity_endpoints:
    toxicity_flags[endpoint + "_flag"] = (
        pd.to_numeric(admet_df[endpoint], errors="coerce") >= 0.5
    )

cyp_flags = candidate_info.copy()

for endpoint in cyp_endpoints:
    cyp_flags[endpoint + "_flag"] = (
        pd.to_numeric(admet_df[endpoint], errors="coerce") >= 0.5
    )

# Export
toxicity_table.to_excel(
    "PAPSS2_Old_Top10_Toxicity_Predictions.xlsx", index=False
)

cyp_table.to_excel(
    "PAPSS2_Old_Top10_CYP_Predictions.xlsx", index=False
)

absorption_table.to_excel(
    "PAPSS2_Old_Top10_Absorption_Predictions.xlsx", index=False
)

toxicity_flags.to_excel(
    "PAPSS2_Old_Top10_Toxicity_Screening_Flags.xlsx", index=False
)

cyp_flags.to_excel(
    "PAPSS2_Old_Top10_CYP_Screening_Flags.xlsx", index=False
)

print("TOXICITY PREDICTIONS")
display(toxicity_table.round(3))

print("CYP PREDICTIONS")
display(cyp_table.round(3))

print("ABSORPTION PREDICTIONS")
display(absorption_table.round(3))

print("Screening flags use a provisional 0.5 probability threshold.")
print("They are not validated toxicity or clinical decision cutoffs.")
print("All five workbooks saved.")

TOXICITY PREDICTIONS


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,AMES,Carcinogens_Lagunin,ClinTox,DILI,Skin_Reaction,hERG,NR-AR-LBD,...,NR-AhR,NR-Aromatase,NR-ER-LBD,NR-ER,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
[H]C1([H])N2C([H])([H])N3C([H])([H])N(C([H])([H])N1S3(=O)=O)S2(=O)=O,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",0.999,0.657,0.074,0.492,0.975,0.071,0.124,...,0.002,0.009,0.043,0.129,0.021,0.700,0.086,0.089,0.016,0.432
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])([H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,0.116,0.553,0.352,0.622,0.445,0.791,0.023,...,0.076,0.205,0.122,0.245,0.394,0.455,0.042,0.115,0.311,0.131
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])(C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",0.063,0.515,0.222,0.702,0.384,0.929,0.008,...,0.038,0.252,0.105,0.220,0.450,0.657,0.019,0.238,0.582,0.163
[H]C([H])(N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H])C([H])([H])C([H])([H])N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H],455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,0.016,0.032,0.092,0.000,0.967,0.699,0.000,...,0.000,0.000,0.001,0.029,0.000,0.008,0.000,0.000,0.001,0.000
[H]O[C@]([H])(C#CC#C[C@]([H])(O[H])C([H])=C([H])[H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H],5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",0.257,0.192,0.035,0.262,0.848,0.267,0.222,...,0.039,0.182,0.370,0.197,0.622,0.739,0.319,0.691,0.491,0.495
[H]O[C@@]([H])(C#CC#CC([H])([H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])=C([H])[H],5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",0.234,0.184,0.021,0.282,0.905,0.300,0.164,...,0.072,0.182,0.349,0.208,0.460,0.757,0.281,0.796,0.474,0.483
[H]C(=C(C([H])([H])[H])C([H])([H])[H])C([H])([H])N=C(N([H])[H])N([H])[H],10983,7,2-(3-methylbut-2-enyl)guanidine,0.201,0.273,0.006,0.008,0.888,0.133,0.007,...,0.011,0.004,0.012,0.041,0.000,0.071,0.001,0.003,0.009,0.002
[H]OC([H])(C([H])([H])C1=C([H])C([H])(C([H])([H])[H])OC1=O)C([H])([H])C([H])([H])C([H])([H])C([H])([H])C1([H])OC([H])(C([H])(O[H])C([H])([H])C([H])([H])C([H])(O[H])C([H])(O[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C1([H])[H],393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",0.193,0.085,0.082,0.330,0.592,0.727,0.017,...,0.016,0.136,0.046,0.063,0.094,0.402,0.015,0.139,0.261,0.071
[Cl-].[H]c1c([H])c([H])c(C(=O)N(C(=O)OC([H])([H])C([H])(OC([H])([H])[H])C([H])([H])OC(=O)N2C([H])([H])C([H])([H])C([H])(OC(=O)N([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C2([H])[H])C([H])([H])c2c([H])c([H])c([H])c([H])[n+]2C([H])([H])C([H])([H])[H])c(OC([H])([H])[H])c1[H],9919108,9,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,0.054,0.293,0.185,0.547,0.511,0.985,0.011,...,0.011,0.304,0.050,0.067,0.255,0.853,0.007,0.543,0.460,0.254
[H]N=C(c1c([H])c([H])c([H])c(C([H])([H])[C@@]([H])(C(=O)N2C([H])([H])C([H])([H])C([H])(C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])C([H])([H])C2([H])[H])N([H])S(=O)(=O)c2c([H])c([H])c(N([H])C(=O)C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])c([H])c2[H])c1[H])N([H])[H],11628768,10,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,0.229,0.054,0.003,0.004,0.652,0.449,0.009,...,0.001,0.003,0.018,0.126,0.00

CYP PREDICTIONS


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,CYP1A2_Veith,CYP2C19_Veith,CYP2C9_Veith,CYP2D6_Veith,CYP3A4_Veith,CYP2C9_Substrate_CarbonMangels,CYP2D6_Substrate_CarbonMangels,CYP3A4_Substrate_CarbonMangels
[H]C1([H])N2C([H])([H])N3C([H])([H])N(C([H])([H])N1S3(=O)=O)S2(=O)=O,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",0.013,0.025,0.002,0.013,0.001,0.055,0.030,0.642
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])([H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,0.172,0.613,0.563,0.598,0.590,0.413,0.132,0.630
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])(C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",0.032,0.682,0.757,0.562,0.835,0.384,0.092,0.779
[H]C([H])(N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H])C([H])([H])C([H])([H])N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H],455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,0.000,0.000,0.000,0.445,0.000,0.001,0.739,0.046
[H]O[C@]([H])(C#CC#C[C@]([H])(O[H])C([H])=C([H])[H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H],5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",0.614,0.321,0.359,0.116,0.336,0.196,0.141,0.649
[H]O[C@@]([H])(C#CC#CC([H])([H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])=C([H])[H],5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",0.931,0.634,0.621,0.192,0.466,0.136,0.098,0.647
[H]C(=C(C([H])([H])[H])C([H])([H])[H])C([H])([H])N=C(N([H])[H])N([H])[H],10983,7,2-(3-methylbut-2-enyl)guanidine,0.024,0.007,0.001,0.121,0.001,0.012,0.504,0.062
[H]OC([H])(C([H])([H])C1=C([H])C([H])(C([H])([H])[H])OC1=O)C([H])([H])C([H])([H])C([H])([H])C([H])([H])C1([H])OC([H])(C([H])(O[H])C([H])([H])C([H])([H])C([H])(O[H])C([H])(O[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C1([H])[H],393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",0.030,0.521,0.365,0.110,0.736,0.141,0.178,0.800
[Cl-].[H]c1c([H])c([H])c(C(=O)N(C(=O)OC([H])([H])C([H])(OC([H])([H])[H])C([H])([H])OC(=O)N2C([H])([H])C([H])([H])C([H])(OC(=O)N([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C2([H])[H])C([H])([H])c2c([H])c([H])c([H])c([H])[n+]2C([H])([H])C([H])([H])[H])c(OC([H])([H])[H])c1[H],9919108,9,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,0.004,0.225,0.311,0.516,0.915,0.106,0.129,0.897
[H]N=C(c1c([H])c([H])c([H])c(C([H])([H])[C@@]([H])(C(=O)N2C([H])([H])C([H])([H])C([H])(C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])C([H])([H])C2([H])[H])N([H])S(=O)(=O)c2c([H])c([H])c(N([H])C(=O)C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])c([H])c2[H])c1[H])N([H])[H],11628768,10,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,0.000,0.005,0.001,0.145,0.061,0.000,0.066,0.011


ABSORPTION PREDICTIONS


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,BBB_Martins,Bioavailability_Ma,HIA_Hou,PAMPA_NCATS,Pgp_Broccatelli
[H]C1([H])N2C([H])([H])N3C([H])([H])N(C([H])([H])N1S3(=O)=O)S2(=O)=O,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",0.955,0.743,0.833,0.604,0.001
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])([H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,0.540,0.655,0.887,0.459,0.754
[H]OP(=O)(O[H])OC([H])([H])[C@]([H])(N([H])C(=O)N1C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]1([H])C(=O)OC([H])(C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])C([H])([H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H])C([H])([H])c1c([H])c([H])c([H])c([H])c1[H],11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",0.519,0.559,0.991,0.553,0.978
[H]C([H])(N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H])C([H])([H])C([H])([H])N1C([H])([H])C([H])([H])C([H])(N2C([H])([H])C([H])([H])C([H])([H])C([H])([H])C2([H])[H])C([H])([H])C1([H])[H],455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,0.866,0.200,0.805,0.612,0.026
[H]O[C@]([H])(C#CC#C[C@]([H])(O[H])C([H])=C([H])[H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H],5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",0.608,0.467,0.986,0.898,0.104
[H]O[C@@]([H])(C#CC#CC([H])([H])/C([H])=C(/[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])=C([H])[H],5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",0.784,0.341,1.000,0.931,0.077
[H]C(=C(C([H])([H])[H])C([H])([H])[H])C([H])([H])N=C(N([H])[H])N([H])[H],10983,7,2-(3-methylbut-2-enyl)guanidine,0.632,0.606,0.804,0.602,0.001
[H]OC([H])(C([H])([H])C1=C([H])C([H])(C([H])([H])[H])OC1=O)C([H])([H])C([H])([H])C([H])([H])C([H])([H])C1([H])OC([H])(C([H])(O[H])C([H])([H])C([H])([H])C([H])(O[H])C([H])(O[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C1([H])[H],393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",0.877,0.571,1.000,0.957,0.783
[Cl-].[H]c1c([H])c([H])c(C(=O)N(C(=O)OC([H])([H])C([H])(OC([H])([H])[H])C([H])([H])OC(=O)N2C([H])([H])C([H])([H])C([H])(OC(=O)N([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C2([H])[H])C([H])([H])c2c([H])c([H])c([H])c([H])[n+]2C([H])([H])C([H])([H])[H])c(OC([H])([H])[H])c1[H],9919108,9,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,0.601,0.279,0.998,0.630,0.996
[H]N=C(c1c([H])c([H])c([H])c(C([H])([H])[C@@]([H])(C(=O)N2C([H])([H])C([H])([H])C([H])(C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])C([H])([H])C2([H])[H])N([H])S(=O)(=O)c2c([H])c([H])c(N([H])C(=O)C([H])([H])C([H])([H])N=C(N([H])[H])N([H])[H])c([H])c2[H])c1[H])N([H])[H],11628768,10,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,0.025,0.095,0.010,0.007,0.001


Screening flags use a provisional 0.5 probability threshold.
They are not validated toxicity or clinical decision cutoffs.
All five workbooks saved.


In [7]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 7: ADMET REGRESSION ENDPOINTS
# ============================================================

import pandas as pd

# Remove inherited SMILES index from ADMET-AI output
admet_df = admet_df.reset_index(drop=True)

regression_endpoints = [
    "Caco2_Wang",
    "Clearance_Hepatocyte_AZ",
    "Clearance_Microsome_AZ",
    "Half_Life_Obach",
    "HydrationFreeEnergy_FreeSolv",
    "LD50_Zhu",
    "Lipophilicity_AstraZeneca",
    "PPBR_AZ",
    "Solubility_AqSolDB",
    "VDss_Lombardo"
]

metadata_cols = [
    "PUBCHEM CID",
    "GLOBAL RANK",
    "COMPOUND NAME"
]

regression_data = admet_df[metadata_cols + regression_endpoints].copy()

# Corresponding DrugBank-approved percentiles
percentile_data = admet_df[
    metadata_cols + [
        endpoint + "_drugbank_approved_percentile"
        for endpoint in regression_endpoints
    ]
].copy()

print("REGRESSION PREDICTIONS")
display(regression_data.round(4))

print("CORRESPONDING REFERENCE PERCENTILES")
display(percentile_data.round(3))

# Export
regression_data.to_excel(
    "PAPSS2_Old_Top10_ADMET_Regression_Predictions.xlsx",
    index=False
)

percentile_data.to_excel(
    "PAPSS2_Old_Top10_ADMET_Regression_Percentiles.xlsx",
    index=False
)

print("\nRegression endpoint audit completed.")
print("Saved both regression workbooks.")

REGRESSION PREDICTIONS


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,Caco2_Wang,Clearance_Hepatocyte_AZ,Clearance_Microsome_AZ,Half_Life_Obach,HydrationFreeEnergy_FreeSolv,LD50_Zhu,Lipophilicity_AstraZeneca,PPBR_AZ,Solubility_AqSolDB,VDss_Lombardo
0,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",-4.9899,43.875801,45.152901,-4.361200,-9.394100,2.4806,0.4405,46.194801,-1.4423,-4.0592
1,44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,-5.8947,43.497101,58.246201,-2.782000,-16.843800,2.5895,2.5029,96.542603,-2.7297,0.9611
2,11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",-5.9118,42.096500,72.146103,17.640699,-16.689199,2.8579,3.3221,102.364700,-3.1916,-0.5440
3,455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,-5.4115,-7.152700,-19.598200,42.908699,-13.836200,2.9130,-0.8158,4.126000,-1.0437,13.1507
4,5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",-4.6987,25.841299,18.037399,36.605999,-9.802400,3.1425,2.3831,88.385696,-3.6022,8.5203
5,5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",-4.3940,61.935299,46.286201,56.984501,-6.552500,3.1031,3.4011,93.915100,-4.9687,11.0154
6,10983,7,2-(3-methylbut-2-enyl)guanidine,-5.1912,26.376900,2.364100,-10.009300,-12.359700,2.3944,-0.3015,19.847401,-0.8449,-1.9630
7,393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",-4.8446,83.612801,61.031700,24.769501,-14.702200,2.1539,2.1773,67.414398,-3.5865,7.2012
8,9919108,9,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,-5.3537,103.729599,158.106903,82.777496,-15.808300,3.1729,3.4514,95.050499,-3.5655,19.0084
9,11628768,10,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,-8.7721,-5.519900,68.252800,4.625100,-44.951500,2.0241,-1.9808,2.473500,-2.4737,5.0660


CORRESPONDING REFERENCE PERCENTILES


,PUBCHEM CID,GLOBAL RANK,COMPOUND NAME,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,64148,1,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",46.362,59.754,65.483,15.255,74.376,52.724,37.188,25.132,78.137,5.659
1,44263409,2,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,16.098,59.438,73.954,17.540,26.292,59.332,71.142,81.933,56.977,45.097
2,11181229,3,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",15.747,58.664,81.336,54.060,26.573,75.220,85.378,92.619,48.049,30.264
3,455458,4,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,26.151,10.545,8.506,77.540,42.636,78.348,19.684,3.374,82.812,94.060
4,5281148,5,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",66.714,44.499,42.531,73.673,72.091,86.960,68.506,66.854,40.176,86.503
5,5281149,6,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",87.909,70.896,66.151,84.464,89.666,85.694,86.924,76.485,14.938,91.634
6,10983,7,2-(3-methylbut-2-enyl)guanidine,36.239,44.956,26.538,7.838,51.845,46.467,25.835,8.858,85.483,17.750
7,393472,8,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",56.134,81.406,75.641,63.902,36.942,31.845,65.378,43.339,40.492,82.742
8,9919108,9,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,28.366,89.209,97.926,92.337,30.439,87.522,87.944,78.981,41.019,98.594
9,11628768,10,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,2.039,11.459,79.930,30.369,2.953,24.745,8.752,2.953,62.074,74.692



Regression endpoint audit completed.
Saved both regression workbooks.


In [8]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 8: LITERATURE EVIDENCE RETRIEVAL
# ============================================================

import requests
import pandas as pd
import time

from urllib.parse import quote

literature_results = []

for _, row in df.iterrows():

    cid = int(row["PUBCHEM CID"])
    compound_name = str(row["IUPAC NAME"])

    queries = [
        f'"{cid}"',
        f'"{compound_name}"'
    ]

    for query in queries:

        url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

        params = {
            "query": query,
            "format": "json",
            "pageSize": 10,
            "resultType": "core"
        }

        try:
            response = requests.get(
                url,
                params=params,
                timeout=30
            )
            response.raise_for_status()

            records = response.json().get("resultList", {}).get("result", [])

            for article in records:

                literature_results.append({
                    "PUBCHEM CID": cid,
                    "COMPOUND NAME": compound_name,
                    "SEARCH QUERY": query,
                    "TITLE": article.get("title", ""),
                    "AUTHOR": article.get("authorString", ""),
                    "YEAR": article.get("pubYear", ""),
                    "JOURNAL": article.get("journalTitle", ""),
                    "DOI": article.get("doi", ""),
                    "PMID": article.get("pmid", ""),
                    "ABSTRACT": article.get("abstractText", ""),
                    "SOURCE": "Europe PMC"
                })

        except Exception as e:
            print(f"Search issue for CID {cid}: {e}")

        time.sleep(0.2)

literature_df = pd.DataFrame(literature_results)

if not literature_df.empty:
    literature_df = literature_df.drop_duplicates(
        subset=["PUBCHEM CID", "TITLE", "DOI", "PMID"]
    )

    literature_df.to_excel(
        "PAPSS2_Old_Top10_Literature_Retrieval.xlsx",
        index=False
    )

    print("Unique literature records:", len(literature_df))
    print("Candidates with search results:",
          literature_df["PUBCHEM CID"].nunique())

    display(
        literature_df[
            ["PUBCHEM CID", "COMPOUND NAME", "TITLE",
             "YEAR", "DOI", "PMID"]
        ]
    )

else:
    print("No literature records retrieved.")

print("Literature retrieval cell completed.")

Unique literature records: 66
Candidates with search results: 9


,PUBCHEM CID,COMPOUND NAME,TITLE,YEAR,DOI,PMID
0,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Oxalate induces phosphatidylserine externaliza...,2026,10.21203/rs.3.rs-9282701/v1,
1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Liposomal Oncolytic Adenovirus Enables Efficie...,2025,10.1021/acsami.5c15766,41231008
2,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",A small lipidated peptide that inhibits transf...,2026,,
3,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Cardiovascular Gaucher Disease Type 3c Associa...,2025,10.1002/ajmg.a.64148,40491257
4,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Anionic Reactions Degrading SF6 Using Metals: ...,2024,10.22541/au.172670600.05743695/v1,
...,...,...,...,...,...,...
61,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",Consensus higher order repeats and frequency o...,2007,10.2174/138920207780368169,18660848
62,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,Cross-Domain Object Detection by Dual Adaptive...,2023,10.3390/s23031199,36772242
63,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,Anxiety and the adult cancer patient.,1998,10.1046/j.1365-2354.1998.00101.x,9919108
64,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,Evaluation of Safety and Efficacy of Emicizuma...,2024,10.4274/tjh.galenos.2024.2024.0220,39169683


Literature retrieval cell completed.


In [9]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 9: LITERATURE MATCH QUALITY AUDIT
# ============================================================

import pandas as pd

# Separate numeric CID searches from compound-name searches
cid_search_records = literature_df[
    literature_df["SEARCH QUERY"].str.match(
        r'"\d+"', na=False
    )
].copy()

name_search_records = literature_df[
    ~literature_df["SEARCH QUERY"].str.match(
        r'"\d+"', na=False
    )
].copy()

print("Total unique records:", len(literature_df))
print("Numeric CID search records:", len(cid_search_records))
print("Compound-name search records:", len(name_search_records))

# Retain name-search results for relevance review
name_search_records = name_search_records.drop_duplicates(
    subset=["PUBCHEM CID", "TITLE", "DOI", "PMID"]
)

# Add manual review fields
name_search_records["EVIDENCE_CLASS"] = "PENDING MANUAL REVIEW"
name_search_records["EXACT_COMPOUND_CONFIRMED"] = "NOT YET VERIFIED"
name_search_records["PAPSS2_RELEVANCE"] = "NOT YET VERIFIED"

# Display records with abstracts for inspection
display(
    name_search_records[
        [
            "PUBCHEM CID",
            "COMPOUND NAME",
            "TITLE",
            "YEAR",
            "DOI",
            "PMID",
            "ABSTRACT"
        ]
    ].reset_index(drop=True)
)

# Save both audit files
cid_search_records.to_excel(
    "PAPSS2_Old_Literature_CID_Search_Audit.xlsx",
    index=False
)

name_search_records.to_excel(
    "PAPSS2_Old_Literature_Name_Search_Review.xlsx",
    index=False
)

print("\nSaved:")
print("1. PAPSS2_Old_Literature_CID_Search_Audit.xlsx")
print("2. PAPSS2_Old_Literature_Name_Search_Review.xlsx")

Total unique records: 66
Numeric CID search records: 61
Compound-name search records: 5


,PUBCHEM CID,COMPOUND NAME,TITLE,YEAR,DOI,PMID,ABSTRACT
0,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",Natural Product-Based Therapeutic Potential of...,2026,10.3390/ph19091330,42797376,<b>Background/Objectives</b>: The increasing g...
1,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",Effect of the dietary polyacetylenes falcarino...,2018,10.1186/s13104-018-3527-y,29945666,<h4>Objectives</h4>(3R)-Falcarinol (FaOH) and ...
2,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol","Terpenoids, Cannabimimetic Ligands, beyond the...",2020,10.3390/molecules25071567,32235333,Medicinal use of <i>Cannabis sativa</i> L. has...
3,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol","Ethnopharmacology, Phytochemistry, and Pharmac...",2019,10.1155/2019/1253493,31915441,"<i>Glehnia littoralis</i> Fr. Schmidt ex Miq, ..."
4,10983,2-(3-methylbut-2-enyl)guanidine,Galegine Content in Goatsrue (Galega officinal...,2011,10.1614/ws-d-10-00169.1,,"Goatsrue is a member of the Fabaceae family, n..."



Saved:
1. PAPSS2_Old_Literature_CID_Search_Audit.xlsx
2. PAPSS2_Old_Literature_Name_Search_Review.xlsx


In [10]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 10: PUBCHEM SYNONYM RETRIEVAL
# ============================================================

import requests
import pandas as pd
import time

synonym_records = []

for cid in df["PUBCHEM CID"].astype(int):

    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/cid/{cid}/synonyms/JSON"
    )

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json()

        info = data["InformationList"]["Information"][0]
        synonyms = info.get("Synonym", [])

        for synonym in synonyms:
            synonym_records.append({
                "PUBCHEM CID": cid,
                "SYNONYM": synonym
            })

        print(f"CID {cid}: {len(synonyms)} synonyms retrieved")

    except Exception as e:
        print(f"CID {cid}: Retrieval failed — {e}")

    time.sleep(0.25)

synonym_df = pd.DataFrame(synonym_records)

synonym_df.to_excel(
    "PAPSS2_Old_Top10_PubChem_Synonyms.xlsx",
    index=False
)

print("\nTotal synonym records:", len(synonym_df))
print("Candidates represented:", synonym_df["PUBCHEM CID"].nunique())

display(synonym_df)

print("Saved: PAPSS2_Old_Top10_PubChem_Synonyms.xlsx")

CID 64148: 40 synonyms retrieved
CID 44263409: 3 synonyms retrieved
CID 11181229: 3 synonyms retrieved
CID 455458: 4 synonyms retrieved
CID 5281148: 48 synonyms retrieved
CID 5281149: 50 synonyms retrieved
CID 10983: 38 synonyms retrieved
CID 393472: 11 synonyms retrieved
CID 9919108: 9 synonyms retrieved
CID 11628768: 4 synonyms retrieved

Total synonym records: 210
Candidates represented: 10


,PUBCHEM CID,SYNONYM
0,64148,Tetramethylenedisulfotetramine
1,64148,TETS
2,64148,80-12-6
3,64148,Tetramethylenedisulphotetramine
4,64148,DSTA
...,...,...
205,9919108,1-Ethyl-2-((2-methoxy-N-((2-methoxy-3-((4-((oc...
206,11628768,CHEMBL380221
207,11628768,BDBM23913
208,11628768,"3-amidinophenylalanine deriv., 56"


Saved: PAPSS2_Old_Top10_PubChem_Synonyms.xlsx


In [11]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 11: SYNONYM-BASED LITERATURE SEARCH
# ============================================================

import requests
import pandas as pd
import re
import time

def usable_synonym(s):
    s = str(s).strip()

    if len(s) < 4 or len(s) > 100:
        return False

    if re.fullmatch(r"\d{2,7}-\d{2}-\d", s):
        return False

    if re.fullmatch(r"(CID|CHEMBL|BDBM|DB\d+)[-_ ]?\w*", s, re.I):
        return False

    if not re.search(r"[A-Za-z]", s):
        return False

    return True


search_synonyms = {}

for cid, group in synonym_df.groupby("PUBCHEM CID"):

    candidates = group["SYNONYM"].dropna().astype(str).tolist()

    # Prefer short, searchable chemical/common names
    candidates = [
        s for s in candidates
        if usable_synonym(s)
    ]

    candidates = sorted(set(candidates), key=len)

    search_synonyms[int(cid)] = candidates[:4]


print("Selected search synonyms:")
for cid, names in search_synonyms.items():
    print(cid, ":", names)


synonym_literature = []

for cid, names in search_synonyms.items():

    compound_name = df.loc[
        df["PUBCHEM CID"].astype(int) == cid,
        "IUPAC NAME"
    ].iloc[0]

    for synonym in names:

        query = f'TITLE_ABS:"{synonym}"'

        params = {
            "query": query,
            "format": "json",
            "pageSize": 10,
            "resultType": "core"
        }

        try:
            response = requests.get(
                "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                params=params,
                timeout=30
            )
            response.raise_for_status()

            records = response.json().get(
                "resultList", {}
            ).get("result", [])

            for article in records:
                synonym_literature.append({
                    "PUBCHEM CID": cid,
                    "IUPAC NAME": compound_name,
                    "SEARCH SYNONYM": synonym,
                    "TITLE": article.get("title", ""),
                    "YEAR": article.get("pubYear", ""),
                    "JOURNAL": article.get("journalTitle", ""),
                    "DOI": article.get("doi", ""),
                    "PMID": article.get("pmid", ""),
                    "ABSTRACT": article.get("abstractText", "")
                })

        except Exception as e:
            print(f"Search error for CID {cid}, {synonym}: {e}")

        time.sleep(0.2)


synonym_literature_df = pd.DataFrame(synonym_literature)

if not synonym_literature_df.empty:

    synonym_literature_df = synonym_literature_df.drop_duplicates(
        subset=["PUBCHEM CID", "TITLE", "DOI", "PMID"]
    )

    synonym_literature_df.to_excel(
        "PAPSS2_Old_Top10_Synonym_Literature.xlsx",
        index=False
    )

    print("\nUnique synonym-based records:",
          len(synonym_literature_df))

    print("Candidates represented:",
          synonym_literature_df["PUBCHEM CID"].nunique())

    display(
        synonym_literature_df[
            [
                "PUBCHEM CID",
                "SEARCH SYNONYM",
                "TITLE",
                "YEAR",
                "DOI",
                "PMID"
            ]
        ]
    )

else:
    print("No synonym-based literature records retrieved.")

print("\nSynonym literature search completed.")

Selected search synonyms:
10983 : ['C08303', 'Galegin', 'Galegine', 'Q27106696']
64148 : ['DSTA', 'TETS', 'ZHA48898', 'Q7706590']
393472 : ['NSC695405', 'NSC-695405', 'ACETOGENINS', 'NCI60_034160']
455458 : ['Bipiperid Pr-bis deriv.', "1,1'-(1,3-Propanediyl)bis(4,1'-bipiperidine)", '4-(1-piperidyl)-1-[3-[4-(1-piperidyl)-1-piperidyl]propyl]piperidine']
5281148 : ['C08449', 'G60922', 'T5S1285', 'F814337']
5281149 : ['C08450', 'G13266', 'TN2360', 'Q413192']
9919108 : ['T19534', 'CS-7447', 'DA-63814', 'HY-U00378']
11181229 : ['pipecolate deriv., 12b']
11628768 : ['3-amidinophenylalanine deriv., 56']
44263409 : ['pipecolate deriv., 11']

Unique synonym-based records: 46
Candidates represented: 3


,PUBCHEM CID,SEARCH SYNONYM,TITLE,YEAR,DOI,PMID
0,10983,Galegin,Multi-Class UHPLC-MS/MS Method for Plant Toxin...,2021,10.1055/a-1517-5828,34243208
1,10983,Galegin,Further hypotensive metabolites from Verbesina...,1999,10.1016/s0960-894x(99)00569-7,10576697
2,10983,Galegin,[Guanidinoacetic acid as a precursor of galegi...,1967,,5599837
3,10983,Galegin,[Guanidino-acetic acid as precursor of galegin...,1967,,5623941
4,10983,Galegin,[ARGININE AS THE FIRST STAGE OF GALEGINE IN GA...,1963,10.1002/ardp.19632960805,14111561
5,10983,Galegin,[Investigations and proposals for selecting al...,1968,10.1007/bf00934201,24442136
6,10983,Galegine,Natural product biosynthesis by guanidine pren...,2026,10.1126/science.aeg1821,42912132
7,10983,Galegine,"Galegine, a Bioprivileged Alkaloid from &lt;i&...",2025,10.2174/0115680266333237241212071912,39886793
8,10983,Galegine,Evaluation of suppressor behavior of guanidine...,2024,10.1002/bab.2636,39014863
9,10983,Galegine,Goat’s Rue,2006,,30000876



Synonym literature search completed.


In [12]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 12: LITERATURE EVIDENCE CLASSIFICATION
# ============================================================

import pandas as pd

# Keep only meaningful chemical-name searches
exclude_terms = [
    "DSTA", "TETS", "NSC695405", "NSC-695405",
    "NCI60_034160", "C08303", "C08449", "C08450",
    "G60922", "G13266", "T5S1285", "F814337",
    "TN2360", "Q413192", "Q27106696", "Q7706590",
    "T19534", "CS-7447", "DA-63814", "HY-U00378"
]

clean_literature = synonym_literature_df[
    ~synonym_literature_df["SEARCH SYNONYM"].isin(exclude_terms)
].copy()

# Separate class-level searches
class_terms = ["ACETOGENINS"]

clean_literature["EVIDENCE_CATEGORY"] = "CHEMICAL NAME SEARCH - VERIFY"

clean_literature.loc[
    clean_literature["SEARCH SYNONYM"].isin(class_terms),
    "EVIDENCE_CATEGORY"
] = "CHEMICAL CLASS - NOT EXACT COMPOUND"

clean_literature["EXACT_COMPOUND_VERIFIED"] = "NO"
clean_literature["PAPSS2_RELEVANCE_VERIFIED"] = "NO"
clean_literature["MANUAL_REVIEW_REQUIRED"] = "YES"

# Remove duplicate article records
clean_literature = clean_literature.drop_duplicates(
    subset=["PUBCHEM CID", "TITLE", "DOI", "PMID"]
)

# Save cleaned literature records
clean_literature.to_excel(
    "PAPSS2_Old_Top10_Cleaned_Literature_Evidence.xlsx",
    index=False
)

# Candidate-level literature summary
literature_summary = (
    clean_literature.groupby("PUBCHEM CID")
    .agg(
        COMPOUND_NAME=("IUPAC NAME", "first"),
        SEARCH_RECORDS=("TITLE", "count"),
        YEARS=("YEAR", lambda x: "; ".join(
            sorted(set(str(v) for v in x if pd.notna(v)))
        )),
        EVIDENCE_TYPES=("EVIDENCE_CATEGORY", lambda x: "; ".join(
            sorted(set(x))
        ))
    )
    .reset_index()
)

literature_summary.to_excel(
    "PAPSS2_Old_Top10_Candidate_Literature_Summary.xlsx",
    index=False
)

print("Cleaned literature records:", len(clean_literature))
print("Candidates represented:", literature_summary.shape[0])

display(literature_summary)

print("\nSaved:")
print("1. PAPSS2_Old_Top10_Cleaned_Literature_Evidence.xlsx")
print("2. PAPSS2_Old_Top10_Candidate_Literature_Summary.xlsx")

Cleaned literature records: 26
Candidates represented: 2


,PUBCHEM CID,COMPOUND_NAME,SEARCH_RECORDS,YEARS,EVIDENCE_TYPES
0,10983,2-(3-methylbut-2-enyl)guanidine,16,1963; 1967; 1968; 1999; 2006; 2021; 2022; 2023...,CHEMICAL NAME SEARCH - VERIFY
1,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",10,2026,CHEMICAL CLASS - NOT EXACT COMPOUND



Saved:
1. PAPSS2_Old_Top10_Cleaned_Literature_Evidence.xlsx
2. PAPSS2_Old_Top10_Candidate_Literature_Summary.xlsx


In [13]:
# ============================================================
# PAPSS2 OLD DERIVATIVES
# CELL 13: INTEGRATED CANDIDATE ASSESSMENT
# ============================================================

import pandas as pd

# Base candidate information
integrated = descriptor_df.copy()

# Add ADMET classification predictions
toxicity_columns = [
    "AMES", "Carcinogens_Lagunin", "ClinTox",
    "DILI", "Skin_Reaction", "hERG"
]

for col in toxicity_columns:
    integrated[col] = admet_df[col].values

# Add selected absorption predictions
absorption_columns = [
    "BBB_Martins", "Bioavailability_Ma",
    "HIA_Hou", "PAMPA_NCATS", "Pgp_Broccatelli"
]

for col in absorption_columns:
    integrated[col] = admet_df[col].values

# Add regression predictions
for col in regression_endpoints:
    integrated[col] = admet_df[col].values

# Literature evidence summary
literature_counts = (
    clean_literature.groupby("PUBCHEM CID")
    .size()
    .to_dict()
)

class_literature_counts = (
    clean_literature[
        clean_literature["EVIDENCE_CATEGORY"]
        == "CHEMICAL CLASS - NOT EXACT COMPOUND"
    ]
    .groupby("PUBCHEM CID")
    .size()
    .to_dict()
)

integrated["CHEMICAL_NAME_SEARCH_RECORDS"] = (
    integrated["PUBCHEM CID"]
    .map(literature_counts)
    .fillna(0)
    .astype(int)
)

integrated["CHEMICAL_CLASS_RECORDS"] = (
    integrated["PUBCHEM CID"]
    .map(class_literature_counts)
    .fillna(0)
    .astype(int)
)

integrated["EXACT_COMPOUND_EVIDENCE"] = "Not confirmed"

# Explicit identity alert
integrated["IDENTITY / SAFETY NOTE"] = ""

integrated.loc[
    integrated["PUBCHEM CID"] == 64148,
    "IDENTITY / SAFETY NOTE"
] = "TETS identity; high-toxicity compound; exclude from therapeutic prioritisation."

# Preserve evidence limitations
integrated["DOCKING LIMITATION"] = (
    "Edock score only; no docked pose or interaction validation available."
)

integrated["ADMET LIMITATION"] = (
    "In-silico predictions; experimental validation required."
)

# Sort by original docking rank
integrated = integrated.sort_values("GLOBAL RANK")

# Save
integrated.to_excel(
    "PAPSS2_Old_Top10_Integrated_Candidate_Assessment.xlsx",
    index=False
)

print("Integrated assessment created.")
print("Candidates:", len(integrated))
print("Columns:", len(integrated.columns))

display(
    integrated[
        [
            "GLOBAL RANK",
            "PUBCHEM CID",
            "IUPAC NAME",
            "EDOCK",
            "Molecular Weight",
            "LogP",
            "QED",
            "Lipinski Violations",
            "AMES",
            "DILI",
            "Skin_Reaction",
            "hERG",
            "CHEMICAL_NAME_SEARCH_RECORDS",
            "CHEMICAL_CLASS_RECORDS",
            "EXACT_COMPOUND_EVIDENCE",
            "IDENTITY / SAFETY NOTE"
        ]
    ].round(3)
)

print("Saved: PAPSS2_Old_Top10_Integrated_Candidate_Assessment.xlsx")

Integrated assessment created.
Candidates: 10
Columns: 44


,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,EDOCK,Molecular Weight,LogP,QED,Lipinski Violations,AMES,DILI,Skin_Reaction,hERG,CHEMICAL_NAME_SEARCH_RECORDS,CHEMICAL_CLASS_RECORDS,EXACT_COMPOUND_EVIDENCE,IDENTITY / SAFETY NOTE
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",-84.317,240.266,-2.445,0.471,0,0.999,0.492,0.975,0.071,0,0,Not confirmed,TETS identity; high-toxicity compound; exclude...
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,-83.905,518.547,3.837,0.221,1,0.116,0.622,0.445,0.791,0,0,Not confirmed,
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",-80.231,636.726,6.229,0.126,2,0.063,0.702,0.384,0.929,0,0,Not confirmed,
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,-74.131,376.633,3.277,0.705,0,0.016,0.000,0.967,0.699,0,0,Not confirmed,
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",-65.493,260.377,2.818,0.400,0,0.257,0.262,0.848,0.267,0,0,Not confirmed,
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",-61.873,244.378,3.847,0.391,0,0.234,0.282,0.905,0.300,0,0,Not confirmed,
6,7,10983,2-(3-methylbut-2-enyl)guanidine,-61.056,127.191,0.226,0.317,0,0.201,0.008,0.888,0.133,16,0,Not confirmed,
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",-60.697,470.647,3.550,0.190,0,0.193,0.330,0.592,0.727,10,10,Not confirmed,
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,-57.702,861.562,6.389,0.059,2,0.054,0.547,0.511,0.985,0,0,Not confirmed,
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,-57.404,627.776,-0.636,0.098,2,0.229,0.004,0.652,0.449,0,0,Not confirmed,


Saved: PAPSS2_Old_Top10_Integrated_Candidate_Assessment.xlsx


In [14]:
# CELL 14: VERIFY INTEGRATED DATA BEFORE PRIORITISATION

print("Integrated assessment shape:", integrated.shape)

print("\nColumn names:")
for col in integrated.columns:
    print(col)

print("\nCandidate identity check:")
display(
    integrated[
        ["GLOBAL RANK", "PUBCHEM CID", "IUPAC NAME", "EDOCK"]
    ].sort_values("GLOBAL RANK")
)

print("\nMissing values:")
display(
    integrated.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

Integrated assessment shape: (10, 44)

Column names:
GLOBAL RANK
PUBCHEM CID
COMPOUND NAME
IUPAC NAME
SOURCE LIBRARY
EDOCK
Molecular Weight
Exact Molecular Weight
LogP
TPSA
HBD
HBA
Rotatable Bonds
Heavy Atoms
Aromatic Rings
QED
Lipinski Violations
AMES
Carcinogens_Lagunin
ClinTox
DILI
Skin_Reaction
hERG
BBB_Martins
Bioavailability_Ma
HIA_Hou
PAMPA_NCATS
Pgp_Broccatelli
Caco2_Wang
Clearance_Hepatocyte_AZ
Clearance_Microsome_AZ
Half_Life_Obach
HydrationFreeEnergy_FreeSolv
LD50_Zhu
Lipophilicity_AstraZeneca
PPBR_AZ
Solubility_AqSolDB
VDss_Lombardo
CHEMICAL_NAME_SEARCH_RECORDS
CHEMICAL_CLASS_RECORDS
EXACT_COMPOUND_EVIDENCE
IDENTITY / SAFETY NOTE
DOCKING LIMITATION
ADMET LIMITATION

Candidate identity check:


,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,EDOCK
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",-84.31694
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,-83.90470
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",-80.23081
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,-74.13148
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",-65.49297
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",-61.87345
6,7,10983,2-(3-methylbut-2-enyl)guanidine,-61.05599
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",-60.69700
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,-57.70152
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,-57.40389



Missing values:


,0
GLOBAL RANK,0
PUBCHEM CID,0
COMPOUND NAME,0
IUPAC NAME,0
SOURCE LIBRARY,0
EDOCK,0
Molecular Weight,0
Exact Molecular Weight,0
LogP,0
TPSA,0


In [15]:
# ============================================================
# CELL 15: OLD DERIVATIVES SCREENING & PRIORITISATION MATRIX
# ============================================================

import pandas as pd
import numpy as np

priority_df = integrated.copy()

# 1. Physicochemical screening
priority_df["MW_WITHIN_500"] = priority_df["Molecular Weight"] <= 500
priority_df["LOGP_WITHIN_5"] = priority_df["LogP"] <= 5
priority_df["LIPINSKI_PASS"] = priority_df["Lipinski Violations"] == 0

# 2. Provisional ADMET screening flags
# These are model-prediction flags, not clinical risk thresholds.
risk_endpoints = [
    "AMES",
    "Carcinogens_Lagunin",
    "ClinTox",
    "DILI",
    "Skin_Reaction",
    "hERG"
]

for col in risk_endpoints:
    priority_df[f"{col}_FLAG"] = priority_df[col] >= 0.5

priority_df["ADMET_FLAG_COUNT"] = priority_df[
    [f"{col}_FLAG" for col in risk_endpoints]
].sum(axis=1)

# 3. Literature evidence classification
priority_df["LITERATURE_STATUS"] = "No relevant records retrieved"

priority_df.loc[
    priority_df["CHEMICAL_NAME_SEARCH_RECORDS"] > 0,
    "LITERATURE_STATUS"
] = "Chemical-name search hits; manual verification required"

priority_df.loc[
    priority_df["CHEMICAL_CLASS_RECORDS"] > 0,
    "LITERATURE_STATUS"
] = "Chemical-class evidence only"

# 4. Explicit exclusion and review categories
priority_df["SCREENING_DECISION"] = "Further evaluation required"

priority_df.loc[
    priority_df["PUBCHEM CID"] == 64148,
    "SCREENING_DECISION"
] = "EXCLUDE: TETS identity and known high toxicity"

priority_df.loc[
    (priority_df["PUBCHEM CID"] != 64148) &
    (priority_df["ADMET_FLAG_COUNT"] >= 3),
    "SCREENING_DECISION"
] = "Caution: multiple ADMET screening flags"

priority_df.loc[
    (priority_df["PUBCHEM CID"] != 64148) &
    (priority_df["ADMET_FLAG_COUNT"] < 3) &
    (priority_df["LIPINSKI_PASS"]) &
    (priority_df["QED"] >= 0.5),
    "SCREENING_DECISION"
] = "Retain for further evaluation"

# 5. Preserve original docking order
priority_df = priority_df.sort_values("GLOBAL RANK")

# 6. Export
priority_df.to_excel(
    "PAPSS2_Old_Top10_Screening_Prioritisation_Matrix.xlsx",
    index=False
)

# 7. Display concise decision matrix
display(
    priority_df[
        [
            "GLOBAL RANK",
            "PUBCHEM CID",
            "IUPAC NAME",
            "EDOCK",
            "QED",
            "Lipinski Violations",
            "ADMET_FLAG_COUNT",
            "LITERATURE_STATUS",
            "SCREENING_DECISION"
        ]
    ].round(3)
)

print("Saved: PAPSS2_Old_Top10_Screening_Prioritisation_Matrix.xlsx")

,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,EDOCK,QED,Lipinski Violations,ADMET_FLAG_COUNT,LITERATURE_STATUS,SCREENING_DECISION
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",-84.317,0.471,0,3,No relevant records retrieved,EXCLUDE: TETS identity and known high toxicity
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,-83.905,0.221,1,3,No relevant records retrieved,Caution: multiple ADMET screening flags
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",-80.231,0.126,2,3,No relevant records retrieved,Caution: multiple ADMET screening flags
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,-74.131,0.705,0,2,No relevant records retrieved,Retain for further evaluation
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",-65.493,0.400,0,1,No relevant records retrieved,Further evaluation required
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",-61.873,0.391,0,1,No relevant records retrieved,Further evaluation required
6,7,10983,2-(3-methylbut-2-enyl)guanidine,-61.056,0.317,0,1,Chemical-name search hits; manual verification...,Further evaluation required
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",-60.697,0.190,0,2,Chemical-class evidence only,Further evaluation required
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,-57.702,0.059,2,3,No relevant records retrieved,Caution: multiple ADMET screening flags
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,-57.404,0.098,2,1,No relevant records retrieved,Further evaluation required


Saved: PAPSS2_Old_Top10_Screening_Prioritisation_Matrix.xlsx


In [16]:
# ============================================================
# CELL 16: FOCUSED REVIEW OF NON-EXCLUDED CANDIDATES
# ============================================================

review_cids = [
    455458, 5281148, 5281149, 10983,
    393472, 11628768, 44263409, 11181229, 9919108
]

review_df = priority_df[
    priority_df["PUBCHEM CID"].isin(review_cids)
].copy()

review_columns = [
    "GLOBAL RANK",
    "PUBCHEM CID",
    "IUPAC NAME",
    "SOURCE LIBRARY",
    "EDOCK",
    "Molecular Weight",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "QED",
    "Lipinski Violations",
    "AMES",
    "Carcinogens_Lagunin",
    "ClinTox",
    "DILI",
    "Skin_Reaction",
    "hERG",
    "Bioavailability_Ma",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Pgp_Broccatelli",
    "LITERATURE_STATUS",
    "SCREENING_DECISION"
]

review_df = review_df[review_columns].sort_values("GLOBAL RANK")

review_df.to_excel(
    "PAPSS2_Old_Top10_Focused_Candidate_Review.xlsx",
    index=False
)

display(review_df.round(3))

print("Candidates reviewed:", len(review_df))
print("Saved: PAPSS2_Old_Top10_Focused_Candidate_Review.xlsx")

,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,SOURCE LIBRARY,EDOCK,Molecular Weight,LogP,TPSA,HBD,HBA,...,ClinTox,DILI,Skin_Reaction,hERG,Bioavailability_Ma,HIA_Hou,PAMPA_NCATS,Pgp_Broccatelli,LITERATURE_STATUS,SCREENING_DECISION
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,Piperidines,-83.905,518.547,3.837,125.40,3,7,...,0.352,0.622,0.445,0.791,0.655,0.887,0.459,0.754,No relevant records retrieved,Caution: multiple ADMET screening flags
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",Piperidines,-80.231,636.726,6.229,125.40,3,7,...,0.222,0.702,0.384,0.929,0.559,0.991,0.553,0.978,No relevant records retrieved,Caution: multiple ADMET screening flags
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,Piperidines,-74.131,376.633,3.277,12.96,0,4,...,0.092,0.000,0.967,0.699,0.200,0.805,0.612,0.026,No relevant records retrieved,Retain for further evaluation
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",Phytochemicals,-65.493,260.377,2.818,40.46,2,2,...,0.035,0.262,0.848,0.267,0.467,0.986,0.898,0.104,No relevant records retrieved,Further evaluation required
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",Phytochemicals,-61.873,244.378,3.847,20.23,1,1,...,0.021,0.282,0.905,0.300,0.341,1.000,0.931,0.077,No relevant records retrieved,Further evaluation required
6,7,10983,2-(3-methylbut-2-enyl)guanidine,Phytochemicals,-61.056,127.191,0.226,64.40,2,1,...,0.006,0.008,0.888,0.133,0.606,0.804,0.602,0.001,Chemical-name search hits; manual verification...,Further evaluation required
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",Phytochemicals,-60.697,470.647,3.550,116.45,4,7,...,0.082,0.330,0.592,0.727,0.571,1.000,0.957,0.783,Chemical-class evidence only,Further evaluation required
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,Piperidines,-57.702,861.562,6.389,136.82,1,9,...,0.185,0.547,0.511,0.985,0.279,0.998,0.630,0.996,No relevant records retrieved,Caution: multiple ADMET screening flags
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,Piperidines,-57.404,627.776,-0.636,274.25,8,7,...,0.003,0.004,0.652,0.449,0.095,0.010,0.007,0.001,No relevant records retrieved,Further evaluation required


Candidates reviewed: 9
Saved: PAPSS2_Old_Top10_Focused_Candidate_Review.xlsx


In [17]:
# ============================================================
# CELL 17: ENDPOINT-LEVEL ADMET RISK AUDIT
# ============================================================

risk_endpoints = [
    "AMES",
    "Carcinogens_Lagunin",
    "ClinTox",
    "DILI",
    "Skin_Reaction",
    "hERG"
]

risk_audit = priority_df[
    ["GLOBAL RANK", "PUBCHEM CID", "IUPAC NAME", "EDOCK"]
].copy()

for endpoint in risk_endpoints:
    risk_audit[endpoint] = priority_df[endpoint].values
    risk_audit[f"{endpoint}_FLAG"] = (
        priority_df[endpoint].values >= 0.5
    )

risk_audit["FLAGGED_ENDPOINTS"] = risk_audit.apply(
    lambda row: "; ".join(
        endpoint for endpoint in risk_endpoints
        if row[f"{endpoint}_FLAG"]
    ) or "None at provisional threshold",
    axis=1
)

risk_audit["RISK_REVIEW_STATUS"] = risk_audit.apply(
    lambda row: (
        "EXCLUDE: TETS identity"
        if row["PUBCHEM CID"] == 64148
        else "Multiple predicted flags"
        if sum(row[f"{endpoint}_FLAG"] for endpoint in risk_endpoints) >= 2
        else "At least one predicted flag"
        if sum(row[f"{endpoint}_FLAG"] for endpoint in risk_endpoints) == 1
        else "No flags at provisional threshold"
    ),
    axis=1
)

risk_audit.to_excel(
    "PAPSS2_Old_Top10_Endpoint_Level_Risk_Audit.xlsx",
    index=False
)

display(
    risk_audit[
        [
            "GLOBAL RANK",
            "PUBCHEM CID",
            "IUPAC NAME",
            *risk_endpoints,
            "FLAGGED_ENDPOINTS",
            "RISK_REVIEW_STATUS"
        ]
    ].round(3)
)

print("Saved: PAPSS2_Old_Top10_Endpoint_Level_Risk_Audit.xlsx")

,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,AMES,Carcinogens_Lagunin,ClinTox,DILI,Skin_Reaction,hERG,FLAGGED_ENDPOINTS,RISK_REVIEW_STATUS
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",0.999,0.657,0.074,0.492,0.975,0.071,AMES; Carcinogens_Lagunin; Skin_Reaction,EXCLUDE: TETS identity
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,0.116,0.553,0.352,0.622,0.445,0.791,Carcinogens_Lagunin; DILI; hERG,Multiple predicted flags
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",0.063,0.515,0.222,0.702,0.384,0.929,Carcinogens_Lagunin; DILI; hERG,Multiple predicted flags
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,0.016,0.032,0.092,0.000,0.967,0.699,Skin_Reaction; hERG,Multiple predicted flags
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",0.257,0.192,0.035,0.262,0.848,0.267,Skin_Reaction,At least one predicted flag
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",0.234,0.184,0.021,0.282,0.905,0.300,Skin_Reaction,At least one predicted flag
6,7,10983,2-(3-methylbut-2-enyl)guanidine,0.201,0.273,0.006,0.008,0.888,0.133,Skin_Reaction,At least one predicted flag
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",0.193,0.085,0.082,0.330,0.592,0.727,Skin_Reaction; hERG,Multiple predicted flags
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,0.054,0.293,0.185,0.547,0.511,0.985,DILI; Skin_Reaction; hERG,Multiple predicted flags
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,0.229,0.054,0.003,0.004,0.652,0.449,Skin_Reaction,At least one predicted flag


Saved: PAPSS2_Old_Top10_Endpoint_Level_Risk_Audit.xlsx


In [18]:
# ============================================================
# CELL 18: FINAL OLD-DERIVATIVE CANDIDATE DISPOSITION
# ============================================================

disposition = priority_df[
    [
        "GLOBAL RANK",
        "PUBCHEM CID",
        "IUPAC NAME",
        "SOURCE LIBRARY",
        "EDOCK",
        "QED",
        "Lipinski Violations",
        "ADMET_FLAG_COUNT",
        "CHEMICAL_NAME_SEARCH_RECORDS",
        "CHEMICAL_CLASS_RECORDS"
    ]
].copy()

disposition["FINAL DISPOSITION"] = "Hold: insufficient evidence"

disposition.loc[
    disposition["PUBCHEM CID"] == 64148,
    "FINAL DISPOSITION"
] = "Exclude: TETS identity and toxicity concern"

disposition.loc[
    disposition["PUBCHEM CID"] == 455458,
    "FINAL DISPOSITION"
] = "Conditional follow-up: investigate hERG and skin-reaction signals"

disposition.loc[
    disposition["PUBCHEM CID"] == 10983,
    "FINAL DISPOSITION"
] = "Literature follow-up: verify exact-compound evidence"

disposition.loc[
    disposition["PUBCHEM CID"] == 393472,
    "FINAL DISPOSITION"
] = "Literature follow-up: exact-compound evidence absent"

disposition["STRUCTURAL_VALIDATION"] = (
    "Required: current SDF contains generated 2D structures, not docked poses"
)

disposition["EXPERIMENTAL_VALIDATION"] = (
    "Required before biological or safety conclusions"
)

disposition = disposition.sort_values("GLOBAL RANK")

disposition.to_excel(
    "PAPSS2_Old_Top10_Final_Candidate_Disposition.xlsx",
    index=False
)

display(disposition)

print("Saved: PAPSS2_Old_Top10_Final_Candidate_Disposition.xlsx")

,GLOBAL RANK,PUBCHEM CID,IUPAC NAME,SOURCE LIBRARY,EDOCK,QED,Lipinski Violations,ADMET_FLAG_COUNT,CHEMICAL_NAME_SEARCH_RECORDS,CHEMICAL_CLASS_RECORDS,FINAL DISPOSITION,STRUCTURAL_VALIDATION,EXPERIMENTAL_VALIDATION
0,1,64148,"2lambda6,6lambda6-dithia-1,3,5,7-tetrazatricyc...",Adamantane,-84.31694,0.470678,0,3,0,0,Exclude: TETS identity and toxicity concern,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
1,2,44263409,4-phenylbutyl (2S)-1-[[(2R)-1-phenyl-3-phospho...,Piperidines,-83.90470,0.221491,1,3,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
2,3,11181229,"1,7-diphenylheptan-4-yl (2S)-1-[[(2R)-1-phenyl...",Piperidines,-80.23081,0.125790,2,3,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
3,4,455458,4-piperidin-1-yl-1-[3-(4-piperidin-1-ylpiperid...,Piperidines,-74.13148,0.705153,0,2,0,0,Conditional follow-up: investigate hERG and sk...,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
4,5,5281148,"(3R,8S,9Z)-heptadeca-1,9-dien-4,6-diyne-3,8-diol",Phytochemicals,-65.49297,0.400102,0,1,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
5,6,5281149,"(3R,9Z)-heptadeca-1,9-dien-4,6-diyn-3-ol",Phytochemicals,-61.87345,0.390506,0,1,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
6,7,10983,2-(3-methylbut-2-enyl)guanidine,Phytochemicals,-61.05599,0.316543,0,1,16,0,Literature follow-up: verify exact-compound ev...,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
7,8,393472,"4-[2-hydroxy-6-[5-(1,4,5-trihydroxyundecyl)oxo...",Phytochemicals,-60.69700,0.189702,0,2,10,10,Literature follow-up: exact-compound evidence ...,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
8,9,9919108,[3-[(1-ethylpyridin-1-ium-2-yl)methyl-(2-metho...,Piperidines,-57.70152,0.058693,2,3,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions
9,10,11628768,N-[4-[[(2S)-3-(3-carbamimidoylphenyl)-1-[4-[2-...,Piperidines,-57.40389,0.097562,2,1,0,0,Hold: insufficient evidence,Required: current SDF contains generated 2D st...,Required before biological or safety conclusions


Saved: PAPSS2_Old_Top10_Final_Candidate_Disposition.xlsx


In [20]:
from pathlib import Path

print("Current working directory:", Path.cwd())

print("\nFiles and folders in /content:")
for path in sorted(Path("/content").iterdir()):
    if path.is_dir():
        print(f"[DIR]  {path.name}/")
    else:
        print(f"[FILE] {path.name} ({path.stat().st_size:,} bytes)")

Current working directory: /content

Files and folders in /content:
[DIR]  .config/
[FILE] PAPSS2_Old_Literature_CID_Search_Audit.xlsx (44,649 bytes)
[FILE] PAPSS2_Old_Literature_Name_Search_Review.xlsx (10,073 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.csv (18,718 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_AI_Full_Predictions.xlsx (17,783 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_Endpoint_Inventory.xlsx (6,655 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_Primary_Predictions.xlsx (12,748 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_Reference_Percentiles.xlsx (10,857 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_Regression_Percentiles.xlsx (7,007 bytes)
[FILE] PAPSS2_Old_Top10_ADMET_Regression_Predictions.xlsx (7,233 bytes)
[FILE] PAPSS2_Old_Top10_Absorption_Predictions.xlsx (6,408 bytes)
[FILE] PAPSS2_Old_Top10_CYP_Predictions.xlsx (6,833 bytes)
[FILE] PAPSS2_Old_Top10_CYP_Screening_Flags.xlsx (5,941 bytes)
[FILE] PAPSS2_Old_Top10_Candidate_Literature_Summary.xlsx (5,218 bytes)
[FILE] PAPSS2_Old_Top10_Clean

In [22]:
from pathlib import Path

# Remove duplicate descriptor workbook from ADMET folder
duplicate = (
    BASE / "04_Molecular_Properties_and_ADMET"
    / "PAPSS2_Old_Top10_RDKit_Descriptors.xlsx"
)

if duplicate.exists():
    duplicate.unlink()

# Create methodology record
methodology = BASE / "08_Methodology" / "PAPSS2_Old_Derivatives_Methodology.txt"

methodology.write_text("""
PAPSS2 OLD DERIVATIVES — COMPUTATIONAL EVALUATION
=================================================

1. INPUT DATA
-------------
PAPSS2_Top10_Old_Derivative.sdf
PAPSS2_Top10_Old_Derivative_docked.xlsx

Ten candidates were supplied for evaluation.
SDF and Excel records were matched using PubChem CID.

2. MOLECULAR DESCRIPTORS
------------------------
RDKit was used to calculate physicochemical and drug-likeness
descriptors, including molecular weight, LogP, TPSA, hydrogen-bond
donors and acceptors, rotatable bonds, aromatic rings and QED.

3. ADMET PREDICTION
-------------------
ADMET-AI was used to generate classification and regression
predictions for the ten candidates.

Predictions are computational estimates and are not experimental
measurements. DrugBank reference percentiles represent relative
positions against the model's reference distribution.

4. LITERATURE RETRIEVAL
----------------------
Europe PMC searches were performed using PubChem CIDs, chemical
names and selected PubChem synonyms.

Numeric CID searches were treated cautiously because PMID/CID
number collisions can produce irrelevant records. Synonym searches
were filtered to reduce acronym and database-identifier false positives.

Search hits were not considered confirmed exact-compound evidence
without manual verification.

5. INTEGRATED ASSESSMENT
------------------------
Docking scores, molecular descriptors, ADMET predictions and
literature-search results were reviewed together.

No composite score was constructed. The provisional 0.5 ADMET
screening threshold is not a validated clinical cutoff.

6. STRUCTURAL LIMITATION
------------------------
The supplied SDF structures are generated 2D structures, not
validated ICM-Pro docked poses. Therefore, protein-ligand interaction
analysis, residue-contact analysis and binding-mode validation
were not performed in this workflow.

7. CANDIDATE DISPOSITION
------------------------
CID 64148 was excluded from therapeutic prioritisation because
its identity is TETS, a highly toxic compound.

CID 455458 was retained only for conditional follow-up because
of predicted hERG and skin-reaction signals.

CID 10983 and CID 393472 were assigned literature follow-up
because their retrieved records did not establish exact-compound
evidence.

No candidate was designated a validated PAPSS2 inhibitor or
therapeutic lead.

8. VALIDATION REQUIREMENTS
--------------------------
Further structural validation, target-specific docking verification,
experimental biological testing and safety assessment are required
before therapeutic conclusions can be made.
""".strip())

print("Methodology file created:", methodology.exists())
print("Duplicate descriptor removed:", not duplicate.exists())
print("Methodology folder contents:")
for p in sorted((BASE / "08_Methodology").iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")

Methodology file created: True
Duplicate descriptor removed: True
Methodology folder contents:
PAPSS2_Old_Derivatives_Methodology.txt (2,597 bytes)


In [26]:
!pip -q install python-docx

In [27]:
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from pathlib import Path

report_dir = BASE / "07_Final_Reports"
report_dir.mkdir(exist_ok=True)

doc = Document()

# Page layout
section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.8)
section.right_margin = Inches(0.8)

# Default font
styles = doc.styles
styles["Normal"].font.name = "Arial"
styles["Normal"].font.size = Pt(9)

# Title
title = doc.add_heading("PAPSS2 Old Derivatives: Top 10 Computational Evaluation", 0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitle = doc.add_paragraph(
    "Integrated molecular-property, ADMET, literature and candidate-disposition assessment"
)
subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(
    "Scope: Evaluation of the ten supplied Old Derivatives candidates. "
    "This report does not represent experimental validation or confirmation "
    "of PAPSS2 inhibitory activity."
)

doc.add_heading("1. Executive Summary", level=1)
doc.add_paragraph(
    "Ten preselected Old Derivatives candidates were evaluated using the supplied "
    "docking-score workbook and SDF structures, RDKit-derived molecular descriptors, "
    "ADMET-AI predictions, literature retrieval and an integrated screening workflow."
)
doc.add_paragraph(
    "CID 64148 was excluded from therapeutic prioritisation because its identity is "
    "tetramethylenedisulfotetramine (TETS), a highly toxic compound. CID 455458 was "
    "retained for conditional follow-up, subject to investigation of its predicted "
    "hERG and skin-reaction signals. CIDs 10983 and 393472 require further literature "
    "verification. The remaining candidates were placed on hold because the current "
    "evidence is insufficient for stronger prioritisation."
)
doc.add_paragraph(
    "No candidate was established as a validated PAPSS2 inhibitor or therapeutic lead."
)

doc.add_heading("2. Input Data and Identity Audit", level=1)
doc.add_paragraph(
    "The supplied input consisted of PAPSS2_Top10_Old_Derivative.sdf and "
    "PAPSS2_Top10_Old_Derivative_docked.xlsx. The workbook contained ten records, "
    "and ten valid SDF structures were identified. All ten PubChem CIDs matched "
    "between the two input files; no CID mismatches were detected."
)
doc.add_paragraph(
    "The SDF records were annotated as generated 2D structures, not ICM-Pro docked "
    "poses. Consequently, the structures cannot support pose-based protein–ligand "
    "interaction analysis."
)

doc.add_heading("3. Molecular Properties and Drug-Likeness", level=1)
doc.add_paragraph(
    "RDKit descriptors were calculated for molecular weight, LogP, TPSA, hydrogen-bond "
    "donors and acceptors, rotatable bonds, aromatic rings, QED and Lipinski violations."
)
doc.add_paragraph(
    "CID 455458 showed the highest QED among the ten candidates (0.705) and zero "
    "Lipinski violations. This is a descriptor-level observation, not evidence of "
    "safety, target activity or therapeutic suitability. CID 9919108 had a low QED "
    "(0.059), high molecular weight (861.562 Da) and LogP (6.389). CID 11628768 "
    "showed high polarity, including TPSA 274.25 Å² and eight hydrogen-bond donors."
)

doc.add_heading("4. ADMET-AI Evaluation", level=1)
doc.add_paragraph(
    "ADMET-AI generated predictions across 52 primary endpoints and corresponding "
    "DrugBank-reference percentiles. Classification-like outputs and regression "
    "outputs were reviewed separately. These are model predictions, not experimental "
    "measurements or clinical probabilities."
)
doc.add_paragraph(
    "A provisional threshold of 0.5 was used to flag selected classification "
    "endpoints, including AMES, carcinogenicity, ClinTox, DILI, skin reaction and "
    "hERG. This threshold is a screening convention and is not a validated clinical "
    "cutoff. All nine candidates other than the excluded TETS candidate had at least "
    "one flagged prediction."
)

doc.add_heading("5. Notable Candidate-Level Findings", level=1)

table = doc.add_table(rows=1, cols=4)
table.style = "Table Grid"
table.alignment = WD_TABLE_ALIGNMENT.CENTER

headers = ["PubChem CID", "Key observations", "Main concern", "Disposition"]
for cell, value in zip(table.rows[0].cells, headers):
    cell.text = value

rows = [
    ("64148", "Edock −84.317; QED 0.471",
     "TETS identity; AMES 0.999; skin reaction 0.975",
     "Exclude"),
    ("455458", "QED 0.705; zero Lipinski violations",
     "hERG 0.699; skin reaction 0.967; predicted bioavailability 0.200",
     "Conditional follow-up"),
    ("10983", "QED 0.317; 16 literature search hits",
     "Exact-compound literature evidence requires verification; skin reaction 0.888",
     "Literature follow-up"),
    ("393472", "QED 0.190; 10 literature search hits",
     "Retrieved records were chemical-class-level; hERG 0.727",
     "Literature follow-up"),
    ("44263409", "Edock −83.905",
     "DILI 0.622; hERG 0.791",
     "Hold"),
    ("11181229", "Edock −80.231",
     "DILI 0.702; hERG 0.929; high LogP and molecular weight",
     "Hold"),
    ("5281148", "QED 0.400",
     "Skin reaction 0.848",
     "Hold"),
    ("5281149", "QED 0.391",
     "Skin reaction 0.905",
     "Hold"),
    ("9919108", "Edock −57.701",
     "hERG 0.985; QED 0.059; high molecular weight and LogP",
     "Hold"),
    ("11628768", "Low predicted CYP interaction probabilities",
     "Poor predicted absorption; high TPSA and HBD",
     "Hold"),
]

for row in rows:
    cells = table.add_row().cells
    for cell, value in zip(cells, row):
        cell.text = value
        cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

doc.add_heading("6. Literature Evidence", level=1)
doc.add_paragraph(
    "Literature retrieval used PubChem CIDs, chemical names and selected synonyms "
    "with Europe PMC. Numeric CID searches produced potential PMID/CID number "
    "collisions and were not treated as reliable compound matches. Synonym searches "
    "also produced acronym and chemical-class false positives."
)
doc.add_paragraph(
    "CID 10983 (galegine) produced name-based records requiring manual verification. "
    "CID 393472 produced acetogenin-related chemical-class records, which do not "
    "establish publications on the exact compound. These search results should not "
    "be interpreted as confirmed compound-specific evidence."
)

doc.add_heading("7. Integrated Candidate Disposition", level=1)
doc.add_paragraph(
    "The integrated assessment did not identify a candidate with sufficiently "
    "consistent evidence to be designated a validated lead. CID 64148 was excluded. "
    "CID 455458 remains a conditional follow-up candidate only. CIDs 10983 and 393472 "
    "require exact-compound literature verification. The other candidates remain "
    "on hold pending additional evidence."
)

doc.add_heading("8. Methodological Limitations", level=1)
limitations = [
    "The supplied SDF files contain generated 2D structures rather than validated docked poses.",
    "No protein–ligand pose inspection or residue-level interaction analysis was possible.",
    "The receptor identity and docking setup require independent verification before interpreting Edock values as PAPSS2-specific binding evidence.",
    "ADMET-AI predictions are computational estimates and require experimental validation.",
    "The provisional 0.5 screening threshold is not a validated clinical decision boundary.",
    "Literature search hits require manual verification; search counts do not establish exact-compound evidence.",
    "No composite ADMET score was constructed because endpoint directionality and validated weighting criteria were not established."
]
for item in limitations:
    doc.add_paragraph(item, style="List Bullet")

doc.add_heading("9. Recommended Next Steps", level=1)
next_steps = [
    "Verify the receptor identity, preparation and docking protocol associated with the supplied Edock scores.",
    "Obtain or generate validated protein–ligand docking poses before performing interaction analysis.",
    "Investigate the hERG and skin-reaction predictions for CID 455458 before considering further prioritisation.",
    "Manually verify exact-compound literature for CIDs 10983 and 393472.",
    "Use experimental PAPSS2 biochemical or binding assays to establish target-specific activity.",
    "Perform appropriate experimental safety and ADMET validation before therapeutic claims."
]
for item in next_steps:
    doc.add_paragraph(item, style="List Number")

doc.add_heading("10. Overall Conclusion", level=1)
doc.add_paragraph(
    "The Old Derivatives evaluation provides a computational screening record for "
    "the ten supplied candidates. The available evidence supports exclusion of "
    "CID 64148 and limited follow-up of selected candidates, but it does not support "
    "the designation of any candidate as a confirmed PAPSS2 inhibitor or therapeutic "
    "lead. Further target-specific structural and experimental validation is required."
)

doc.add_paragraph(
    "Report generated from the computational outputs stored in the project package."
)

report_path = report_dir / "PAPSS2_Old_Derivatives_Supervisor_Ready_Evaluation_Report.docx"
doc.save(report_path)

print("Report created:", report_path.exists())
print("File size:", f"{report_path.stat().st_size:,} bytes")

Report created: True
File size: 39,794 bytes


In [28]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/PAPSS2_Top10_Old_Derivatives_FINAL.zip")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(BASE.rglob("*")):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(BASE.parent))

# Verify archive integrity and contents
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    file_list = zf.namelist()

print("ZIP created:", ZIP_PATH.exists())
print("ZIP size:", f"{ZIP_PATH.stat().st_size:,} bytes")
print("Files archived:", len(file_list))
print("Integrity check:", "PASS" if bad_file is None else f"FAILED: {bad_file}")

print("\nFolder-wise archive count:")
from collections import Counter
counts = Counter(name.split("/")[1] for name in file_list if "/" in name)

for folder, count in sorted(counts.items()):
    print(f"{folder}: {count} files")

ZIP created: True
ZIP size: 457,577 bytes
Files archived: 31
Integrity check: PASS

Folder-wise archive count:
01_Source_Files: 2 files
02_Data_Quality_and_Structure_Audit: 2 files
03_Docking_Results: 1 files
04_Molecular_Properties_and_ADMET: 12 files
05_Literature_Analysis: 7 files
06_Integrated_Candidate_Assessment: 5 files
07_Final_Reports: 1 files
08_Methodology: 1 files


In [29]:
from google.colab import files
files.download("/content/PAPSS2_Top10_Old_Derivatives_FINAL.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
import pandas as pd
import zipfile
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BASE = Path("/content/PAPSS2_Top10_Old_Derivatives_FINAL")
REPORTS = BASE / "07_Final_Reports"
SELECTED = BASE / "09_Selected_Candidate_Structure"

REPORTS.mkdir(exist_ok=True)
SELECTED.mkdir(exist_ok=True)

# Locate source workbooks
integrated_path = BASE / "06_Integrated_Candidate_Assessment" / \
    "PAPSS2_Old_Top10_Integrated_Candidate_Assessment.xlsx"

disposition_path = BASE / "06_Integrated_Candidate_Assessment" / \
    "PAPSS2_Old_Top10_Final_Candidate_Disposition.xlsx"

assert integrated_path.exists(), "Integrated assessment workbook missing."
assert disposition_path.exists(), "Final disposition workbook missing."

integrated = pd.read_excel(integrated_path)
disposition = pd.read_excel(disposition_path)

# 1. Named candidate comparison workbook
comparison_path = REPORTS / "01_PAPSS2_Final_Named_Candidate_Comparison.xlsx"

with pd.ExcelWriter(comparison_path, engine="openpyxl") as writer:
    integrated.to_excel(writer, sheet_name="Named Candidate Comparison", index=False)

# 2. Named final decision matrix workbook
decision_path = REPORTS / "02_PAPSS2_Final_Decision_Matrix_Named.xlsx"

with pd.ExcelWriter(decision_path, engine="openpyxl") as writer:
    disposition.to_excel(writer, sheet_name="Final Decision Matrix", index=False)
    integrated[[
        c for c in [
            "GLOBAL RANK", "PUBCHEM CID", "COMPOUND NAME",
            "IUPAC NAME", "SOURCE LIBRARY", "EDOCK",
            "QED", "Lipinski Violations"
        ] if c in integrated.columns
    ]].to_excel(writer, sheet_name="Candidate Reference", index=False)

# Apply readable workbook formatting
for path in [comparison_path, decision_path]:
    wb = load_workbook(path)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="1F4E78")
            cell.alignment = Alignment(wrap_text=True, vertical="center")
        for col in ws.columns:
            letter = get_column_letter(col[0].column)
            max_len = max(
                len(str(cell.value or "")) for cell in col[:100]
            )
            ws.column_dimensions[letter].width = min(max(max_len + 2, 12), 45)
    wb.save(path)

# 3. Verify selected-candidate SDF; extract from original if absent
selected_sdf = SELECTED / "PAPSS2_CID_455458_Selected_Candidate.sdf"

if not selected_sdf.exists():
    from rdkit import Chem

    source_sdf = BASE / "01_Source_Files" / "PAPSS2_Top10_Old_Derivative.sdf"
    assert source_sdf.exists(), "Original SDF missing."

    supplier = Chem.SDMolSupplier(str(source_sdf), removeHs=False)
    matches = []

    for mol in supplier:
        if mol is None:
            continue
        props = mol.GetPropsAsDict()
        cid = str(
            props.get("PUBCHEM_CID",
            props.get("PUBCHEM_COMPOUND_CID",
            props.get("PUBCHEM CID", "")))
        )
        if cid == "455458":
            matches.append(mol)

    assert len(matches) == 1, f"Expected one CID 455458 structure; found {len(matches)}."
    writer = Chem.SDWriter(str(selected_sdf))
    writer.write(matches[0])
    writer.close()

# 4. Rebuild final ZIP
zip_path = Path("/content/PAPSS2_Top10_Old_Derivatives_FINAL_COMPLETE.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(BASE.rglob("*")):
        if file.is_file():
            zf.write(file, file.relative_to(BASE.parent))

with zipfile.ZipFile(zip_path) as zf:
    bad = zf.testzip()
    archived = zf.namelist()

print("CREATED:")
print(comparison_path.name)
print(decision_path.name)
print(selected_sdf.name)
print("\nSelected SDF:", selected_sdf.stat().st_size, "bytes")
print("Archive files:", len(archived))
print("ZIP integrity:", "PASS" if bad is None else f"FAIL: {bad}")
print("Final ZIP:", zip_path)

CREATED:
01_PAPSS2_Final_Named_Candidate_Comparison.xlsx
02_PAPSS2_Final_Decision_Matrix_Named.xlsx
PAPSS2_CID_455458_Selected_Candidate.sdf

Selected SDF: 6317 bytes
Archive files: 34
ZIP integrity: PASS
Final ZIP: /content/PAPSS2_Top10_Old_Derivatives_FINAL_COMPLETE.zip


In [31]:
from google.colab import files

files.download(
    "/content/PAPSS2_Top10_Old_Derivatives_FINAL_COMPLETE.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>